# Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

Dataset citation:
> Kulka, M., Rodriguez Miera, S. & Marcet-Palacios, M. 2026. Mass Spectrometry Analysis of Extracellular Vesicles Isolated from Stimulated Human Mast Cells. Frontiers.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the dataset Croissant schema URL
url = 'https://sen.science/doi/10.71728/senscience.vq4a-28xa/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(url)
metadata = dataset.metadata

print(f"Dataset: {metadata.name}")
print(f"Description: {metadata.description}")
print(f"Published: {metadata.datePublished}")
print(f"Version: {metadata.version}")
print(f"Identifier: {metadata.identifier}\n")

## 2. Data Overview
Review available **record sets**, **fields**, and their **@id**s in the Croissant dataset. We will reference all entities by their `@id`.

In [ ]:
from pprint import pprint

# List all record sets in the dataset, showing their @id and fields
record_sets = list(metadata.record_sets)

print("RecordSets (with @id and name):\n")
for rs in record_sets:
    print(f"  @id: {rs.id}  |  name: {rs.name if hasattr(rs, 'name') else '(none)'}")

print("\nRecord set Fields (by RecordSet @id):\n")
for rs in record_sets:
    print(f"RecordSet @id: {rs.id}")
    for f in rs.fields:
        print(f"    Field @id: {f.id}    label: {getattr(f, 'label', f.name) if hasattr(f, 'label') else f.name if hasattr(f, 'name') else ''}")
    print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis. Use the record set and field `@id`s identified above.

In [ ]:
# Select the main record set for protein analysis (find appropriate @id from above)
# We'll assume the main record set is the first listed, but you can change this based on the overview above.
main_record_set_id = record_sets[0].id if len(record_sets) > 0 else None

print(f"Main record set selected: {main_record_set_id}")

# You can extract multiple record sets if needed, by listing their @id values here
record_set_ids = [rs.id for rs in record_sets]
dataframes = {}

for rs_id in record_set_ids:
    records = list(dataset.records(record_set=rs_id))
    if records:
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Loaded record set @id: {rs_id} with shape {dataframes[rs_id].shape}")
    else:
        print(f"Record set @id: {rs_id} yielded no records.")


# Show the columns (Fields) and first few rows for the main record set
if main_record_set_id:
    df = dataframes[main_record_set_id]
    print("\nColumns (fields) in main record set DataFrame:")
    pprint(list(df.columns))
    display(df.head())

## 4. Exploratory Data Analysis (EDA)
Apply data processing steps such as filtering, normalization, and grouping. Reference all fields by their canonical field (column) `@id`.

In [ ]:
# Pick a numeric field for filtering and normalization
# For this dataset, likely fields:
# - e.g., 'cr:field/mw' (Molecular weight), 'cr:field/abundance', etc.
# Find a numeric field in the record set
df = dataframes[main_record_set_id]

# Show candidate numeric field columns by dtype
numeric_cols = df.select_dtypes(include='number').columns.tolist()
print("Numeric columns:", numeric_cols)

# If present, use as an example 'cr:field/mw' or a numeric field
if len(numeric_cols) > 0:
    numeric_field = numeric_cols[0]
else:
    raise RuntimeError("No numeric columns found in main record set.")
print(f"Using numeric field: {numeric_field}")

# Filter records based on a threshold
threshold = df[numeric_field].mean() if df[numeric_field].dtype in [int, float] else 10
filtered_df = df[df[numeric_field] > threshold]
print(f"Filtered records with {numeric_field} > {threshold}:")
display(filtered_df.head())

# Normalize the numeric field
normalized_col = f"{numeric_field}_normalized"
filtered_df[normalized_col] = (filtered_df[numeric_field] - filtered_df[numeric_field].mean()) / filtered_df[numeric_field].std()
print(f"Normalized {numeric_field} (standardized Z-score):")
display(filtered_df[[numeric_field, normalized_col]].head())

# Attempt to group by a categorical field, such as 'cr:field/accession' or another non-numeric column
group_field = None
for col in df.columns:
    if df[col].dtype == 'object' and col != numeric_field:
        group_field = col
        break

if group_field:
    grouped_df = filtered_df.groupby(group_field)[numeric_field].mean()
    print(f"Mean {numeric_field} grouped by {group_field}:")
    display(grouped_df.head())
else:
    print("No suitable string/categorical field found for grouping.")

## 5. Visualization
Visualize data distributions or relationships between fields.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Plot the distribution of the numeric field
plt.figure(figsize=(8, 4))
sns.histplot(df[numeric_field], kde=True)
plt.title(f"Distribution of {numeric_field}")
plt.xlabel(numeric_field)
plt.ylabel("Count")
plt.show()

# If there's a group_field, show box plot
if group_field:
    plt.figure(figsize=(12, 4))
    sns.boxplot(data=filtered_df, x=group_field, y=numeric_field)
    plt.title(f"{numeric_field} by {group_field}")
    plt.xticks(rotation=45)
    plt.show()

## 6. Conclusion
This notebook demonstrated how to explore and analyze the FAIR² mass spectrometry dataset using the `mlcroissant` library. We loaded the dataset via its Croissant schema, listed available record sets and fields (by `@id`), extracted and processed data, and generated basic visualizations. For further exploration, review other fields and record sets using their canonical `@id` references or extend the EDA and modeling with your own processing steps.